# DADA2: 02 — Retrieval strategies

**6 points.** Three TODOs, and one strategy given to you at the end.

You have an index. There are more ways to search it than the one
`similarity_search` call from W05. We'll build them one at a time and measure
each one as soon as it works.

Every strategy has the same signature, and that is not a stylistic
preference. It is what makes the comparison fair:

```
search(question, k) -> [(source_file, passage_text), ...]    best first
```

Two strategies that return different shapes cannot be compared. If you find
yourself wanting to change the contract, change it for all five.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from rag import config, evaluate, retrieval

## 1. How we will measure

Two metrics.

**recall@k**: of the files that answer this question, how many came back in
the top k? This one matters most. A passage the retriever never returned is a
passage the model cannot use, and nothing downstream fixes that.

**MRR**: how high up was the first relevant file? Recall says the evidence was
in the list. MRR says whether it was at the top. They move independently, and
the gap between them is the case for reranking.

`eval/evalset.json` holds 14 questions, each naming a company and a fiscal
year, each labeled with the one file that answers it.

In [2]:
specific = evaluate.load_evalset("specific")

print(f"{len(specific)} questions\n")
for item in specific[:4]:
    print(f"  {item['question']}")
    print(f"    gold: {evaluate.gold_of(item)}\n")

14 questions

  What does MU state that the preparation of financial statements and related disclosures in conformity with U.S. GAAP requires management to do in its FY2024 10-K?
    gold: ['MU_2024_business.txt']

  What termination fee is AMD required to pay ZT Systems under certain circumstances related to the failure to obtain required regulatory approvals according to the FY2024 10-K?
    gold: ['AMD_2024_risk_factors.txt']

  What factors does NVIDIA believe are the principal competitive factors in its market for fiscal year 2025?
    gold: ['NVDA_2025_business.txt']

  What competitive pressures from Intel and Nvidia did AMD face in fiscal year 2025 that could affect its market share and margins?
    gold: ['AMD_2025_risk_factors.txt']



### Pacing, before you run anything

The free Gemini tier allows 15 requests a minute and 500 a day. Scoring one
strategy over 14 questions costs between 14 and 84 calls depending on the
strategy. Therefore, you will waste your time if you try to score all five over both question sets in one afternoon.

So while you are developing, score on a slice. The `SAMPLE` below is six
questions, which is enough to tell a working strategy from a broken one.
Notebook 04 does the full run, once, when everything already works.

`rag/util.py` has a `retry` helper that backs off when the API says you are
over the limit. Every model call you write should go through it.

In [3]:
SAMPLE = specific[:6]
print(f"developing against {len(SAMPLE)} questions")

developing against 6 questions


## 2. The baseline

The strategy you already know. Embed the question as the user typed it, one
search, no filter. It is given to you in `rag/retrieval.py` so there is a
floor to compare against.

In [4]:
question = "What does NVIDIA say about export controls in its fiscal 2026 risk factors?"

for source, text in retrieval.baseline(question, k=5):
    print(f"  {source}")

c:\Users\harsh\AgenticAI_Fall26\DADA2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  NVDA_2025_risk_factors.txt
  NVDA_2025_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2025_risk_factors.txt
  NVDA_2024_risk_factors.txt


Look at the filenames. The question named one filing. How many came back, and
how many are the right company in the wrong year?

The passages are all about export controls. That is the problem. Four years of
NVIDIA and two competitors say similar things, and cosine similarity has no
way to prefer the year you asked for. "Fiscal 2026" is three tokens out of
twenty, averaged into the same vector as everything else.

Put a number on it.

In [5]:
base = evaluate.score_retrieval(retrieval.baseline, SAMPLE, k=5, verbose=True)
print()
print(f"baseline   recall@5 {base['recall@k']:.2f}   MRR {base['mrr']:.3f}")

  [NO ] What does MU state that the preparation of financial statements an
  [yes] What termination fee is AMD required to pay ZT Systems under certa
  [yes] What factors does NVIDIA believe are the principal competitive fac
  [yes] What competitive pressures from Intel and Nvidia did AMD face in f
  [yes] What forms the basis for AMD's estimates and judgments used in pre
  [NO ] What financial charge did NVIDIA incur in fiscal year 2026 due to 

baseline   recall@5 0.67   MRR 0.325


In [6]:
# run this line for interpretation
import math
print("The meaning of recall@5 result: ")
print(f"Out of the queries tested, \n the baseline retriever successfully included a relevant document \n in its top 5 results {base['recall@k']:.1%} of the time.")

implied_rank = 1 / base['mrr']
print("\nThe meaning of MRR result: ")
print(f"The average position of the first relevant document is roughly {implied_rank:.1f}.")


The meaning of recall@5 result: 
Out of the queries tested, 
 the baseline retriever successfully included a relevant document 
 in its top 5 results 66.7% of the time.

The meaning of MRR result: 
The average position of the first relevant document is roughly 3.1.


What the baseline shows is that while the retriever finds the right answer approximately 60-70% of the time, it rarely puts it at position #1 or #2.

## 3. Strategy one: filter on metadata

### TODO 2.1 — pull the filter out of the question (2 pts)

The question carries three separate things: a company, a fiscal year, and a
thing to look for. The embedding treats all three as one soup. Separate them,
hand the first two to Pinecone as a filter, and search for only the third.

In `rag/retrieval.py`, write `Filter`, `extract_filter` and `where_of`, then
`with_filter`.

The docstrings cover the two traps: an empty filter dict is
not the same as no filter, and the company and year must be optional because
plenty of questions name neither.

Write it here, then move it into the module.

**Steps:**

1. `Filter`: a `BaseModel` with three fields. `ticker: Optional[Literal["NVDA", "AMD", "MU"]] = None`, `fiscal_year: Optional[int] = None`, `search: str`. Give each a `Field(description=...)`; the model reads them.
2. `extract_filter(question)`: `model = config.chat().with_structured_output(Filter)`, then `retry(model.invoke, prompt)`. The prompt is the question plus one instruction: pull out the company, the year, and what to search for, and leave company or year empty unless the question names exactly one.
3. `where_of(f)`: start with an empty dict. If `f.ticker`, add `{"ticker": {"$eq": f.ticker}}`. Same for `fiscal_year`. Return the dict if it has anything in it, otherwise `None`.
4. `with_filter(question, k)`: `f = extract_filter(question)`, then `retrieval.dense(f.search, k, where=where_of(f))`. Search for `f.search`, not the question.


In [7]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from rag.util import retry

class Filter(BaseModel):
    """Which filings the question is about, and what to look for in them."""
    # TODO 2.1 — three fields. Two of them are optional, and it matters that
    # they are, because plenty of questions name no company at all.
    # your code here
    ticker: Optional[Literal["NVDA", "AMD", "MU"]] = Field(
        default=None,
        description="Ticker of the one company the question names: NVDA (NVIDIA), "
                    "AMD, or MU (Micron). Null if it names none or more than one.",
    )
    fiscal_year: Optional[int] = Field(
        default=None,
        description="The one fiscal year the question names, as a 4-digit integer "
                    "(e.g. 'FY2024' or 'fiscal 2024' -> 2024). Null if none or more than one.",
    )
    search: str = Field(
        description="What to look for, with the company name and year removed.",
    )



def extract_filter(question):
    # TODO 2.1 — with_structured_output, as in W05b.1. Tell the model to
    # leave company and year empty unless the question names exactly one.
    # your code here
    model = config.chat().with_structured_output(Filter)
    prompt = (
        "From the question below, extract the company, the fiscal year, and what to "
        "search for. Leave company and fiscal year empty unless the question names "
        "exactly one of each.\n\n"
        f"Question: {question}"
    )
    return retry(model.invoke, prompt)


def where_of(f):
    """Turn a Filter into a Pinecone filter dict.

    Pinecone wants operators, so ticker NVDA becomes {"ticker": {"$eq": "NVDA"}}.
    Watch the empty case: {} is not the same as no filter, and passing {} will
    not do what you want. Return None when there is nothing to filter on.
    """
    # TODO 2.1
    # your code here
    where = {}
    if f.ticker:
        where["ticker"] = {"$eq": f.ticker}
    if f.fiscal_year:
        where["fiscal_year"] = {"$eq": f.fiscal_year}
    return where or None



In [8]:
# Does it read the question correctly?
f = extract_filter(question)
print("ticker:     ", f.ticker)
print("fiscal_year:", f.fiscal_year, type(f.fiscal_year).__name__)
print("search:     ", f.search)
print("where:      ", where_of(f))

print()
vague = "How do these companies describe their dependence on TSMC?"
g = extract_filter(vague)
print("on a vague question:", g.ticker, g.fiscal_year, "|", g.search)
print("where:", where_of(g), "   <- must be None, not {}")

ticker:      NVDA
fiscal_year: 2026 int
search:      export controls risk factors
where:       {'ticker': {'$eq': 'NVDA'}, 'fiscal_year': {'$eq': 2026}}

on a vague question: None None | dependence on TSMC
where: None    <- must be None, not {}


NOTE:

A question naming no company has to
produce no filter. If `where_of` returns `{}` there, you have asked Pinecone
to match a filter with no conditions, and what you get back depends on the client rather than on what you meant.

In [9]:
# TODO 2.1 — with_filter, using the three pieces above
def with_filter(question, k=config.K):
    # your code here
    f = extract_filter(question)
    return retrieval.dense(f.search, k, where=where_of(f))

for source, text in with_filter(question, k=5):
    print(f"  {source}")

  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt


In [10]:
filtered = evaluate.score_retrieval(with_filter, SAMPLE, k=5, verbose=True)
print()
print(f"baseline   recall@5 {base['recall@k']:.2f}   MRR {base['mrr']:.3f}")
print(f"filter     recall@5 {filtered['recall@k']:.2f}   MRR {filtered['mrr']:.3f}")

  [yes] What does MU state that the preparation of financial statements an
  [yes] What termination fee is AMD required to pay ZT Systems under certa
  [yes] What factors does NVIDIA believe are the principal competitive fac
  [yes] What competitive pressures from Intel and Nvidia did AMD face in f
  [yes] What forms the basis for AMD's estimates and judgments used in pre
  [yes] What financial charge did NVIDIA incur in fiscal year 2026 due to 

baseline   recall@5 0.67   MRR 0.325
filter     recall@5 1.00   MRR 0.750


On the full 14 questions this takes recall from 0.71 to 1.00. Largest gain in
the notebook, and it is a filter, not anything clever about embeddings.

NOTE:
A filter is a promise that you know which documents
the answer is in. These 14 questions name their company and year, so the
promise holds. In section 6, you will run the same strategy on questions that do
not.

## 4. Strategy two: Multi-query with RRF

### Why one query is not enough

A single phrasing finds what that phrasing happens to sit near in embedding
space. "Export control restrictions" and "trade compliance limits on chip
sales" mean nearly the same thing and land in different neighborhoods. Whichever one you typed is the one you searched with.

The solution to this problem is to ask several ways, which is what've already done in the lectures. Asking multiple questions leaves you with multiple ranked lists, and you need one.

### Reciprocal rank fusion

You cannot average the scores as you may be using both semantic search (cosine similarity between embeddings) and lexical search (BM25). A cosine similarity of 0.82 and a BM25 score
of 11.4 are not on the same scale, and even two cosine scores from differently
filtered searches are not comparable. The one thing every retriever agrees on is **position**.

RRF scores each item by where it ranked, summed across the lists:

$$\text{score}(d) = \sum_{\text{lists}} \frac{1}{k + \text{rank}(d)}$$

with rank counting from 0 and $k = 60$ by convention. The constant flattens
the top of each list so rank 1 and rank 2 are not wildly far apart, which
stops a single confident list from dominating the fusion.

Let's do it by hand on two small lists before writing any code.

In [11]:
# two rankings of four documents
dense_list   = ["X", "Y", "Z"]
lexical_list = ["Y", "W", "X"]

k = 60
scores = {}
for ranking in (dense_list, lexical_list):
    for position, doc in enumerate(ranking):
        contribution = 1 / (k + position + 1)
        scores[doc] = scores.get(doc, 0) + contribution
        print(f"  {doc}: rank {position} in its list -> +1/{k + position + 1} = {contribution:.6f}")
    print()

print("fused:")
for doc in sorted(scores, key=scores.get, reverse=True):
    print(f"  {doc}  {scores[doc]:.6f}")

  X: rank 0 in its list -> +1/61 = 0.016393
  Y: rank 1 in its list -> +1/62 = 0.016129
  Z: rank 2 in its list -> +1/63 = 0.015873

  Y: rank 0 in its list -> +1/61 = 0.016393
  W: rank 1 in its list -> +1/62 = 0.016129
  X: rank 2 in its list -> +1/63 = 0.015873

fused:
  Y  0.032522
  X  0.032266
  W  0.016129
  Z  0.015873


Read the result. **X** was first in one list and third in the other. **Y** was
second and first. Y wins, narrowly, because it was near the top of both lists
while X had one list that loved it and one that did not.

That is the behavior we want. A passage on which several different phrasings all
agree is a better bet than a passage that one phrasing ranked first and the
others did not return at all.

Notice that we didn't do anything with the similarity scores. Only positions.

### TODO 2.2 — multi-query and fusion (2 pts)

In `rag/retrieval.py`, write `Queries`, `write_queries`, `rrf`, then
`filter_rrf`.

`rrf` takes a list of ranked lists and returns one ranked list, best first.
Every item has to be hashable, which is why the strategies pass
`(source, text)` tuples around rather than `Document` objects.

Keep the filter from TODO 2.1 on every one of the searches. We are changing
one thing at a time, and the one thing being changed here is the number of
queries.

**Steps:**

1. `Queries`: a `BaseModel` with one field, `queries: list[str]`.
2. `write_queries(text, n=4)`: structured output again. Ask for `n` different searches that cover the same thing, varied in wording. Return `out.queries[:n]`, or `[text]` if it gave you nothing.
3. `rrf(rankings, k=config.RRF_K)`: a dict of item to score. For each ranking, for each `(position, item)` in `enumerate(ranking)`, add `1 / (k + position + 1)`. Return the items sorted by score, highest first. That is the whole function.
4. `filter_rrf(question, k)`: extract the filter once. One `retrieval.dense` search per rewrite, each with the same `where`. Pass the list of result lists to `rrf` and take the first `k`.


In [12]:
class Queries(BaseModel):
    """Several ways of asking the same thing."""
    # TODO 2.2
    # your code here
    queries: list[str] = Field(
        description="Different search queries that all look for the same information, "
                    "each worded differently.",
    )


def write_queries(text, n=4):
    # TODO 2.2 — ask the model for n searches that cover the same intent.
    # Return the original text as a single-item list if it gives you nothing.
    # your code here
    model = config.chat().with_structured_output(Queries)
    prompt = (
        f"Write {n} different search queries for finding passages in SEC 10-K filings "
        "that cover the same thing as the search below. Keep the meaning the same, but "
        "vary the wording and terminology (synonyms, formal filing language, related "
        "phrasing).\n\n"
        f"Search: {text}"
    )
    out = retry(model.invoke, prompt)
    queries = [q for q in (out.queries if out else []) if q and q.strip()]
    return queries[:n] or [text]



def rrf(rankings, k=config.RRF_K):
    """Fuse ranked lists. Takes a list of lists, returns one list, best first.

    Every item must be hashable, which is why the strategies pass
    (source, text) tuples around rather than Document objects.
    """
    # TODO 2.2
    # your code here
    scores = {}
    for ranking in rankings:
        for position, item in enumerate(ranking):
            scores[item] = scores.get(item, 0) + 1 / (k + position + 1)
    return sorted(scores, key=scores.get, reverse=True)


def filter_rrf(question, k=config.K):
    # TODO 2.2 — filter_rrf
    # your code here
    f = extract_filter(question)
    where = where_of(f)
    rankings = [retrieval.dense(q, k, where=where) for q in write_queries(f.search)]
    return rrf(rankings)[:k]


In [13]:
# what the rewrites look like
f = extract_filter(question)
queries = write_queries(f.search)
for q in queries:
    print(" -", q)

 - risks related to export restrictions and international trade regulations
 - impact of export compliance controls on our business operations
 - regulatory vulnerabilities associated with cross-border shipping and technology transfers
 - export control laws and potential adverse effects on financial condition


In [14]:
# rrf on your own lists, same hand-checked case as above
check = rrf([["X", "Y", "Z"], ["Y", "W", "X"]])
print("your rrf says:", check)
assert check[:2] == ["Y", "X"], f"expected Y then X, got {check[:2]}"
print("matches the hand calculation")

your rrf says: ['Y', 'X', 'W', 'Z']
matches the hand calculation


In [15]:
for source, text in filter_rrf(question, k=5):
    print(f"  {source}")

  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt
  NVDA_2026_risk_factors.txt


In [16]:
fused = evaluate.score_retrieval(filter_rrf, SAMPLE, k=5, verbose=True)
print()
print(f"filter       recall@5 {filtered['recall@k']:.2f}   MRR {filtered['mrr']:.3f}")
print(f"filter+rrf   recall@5 {fused['recall@k']:.2f}   MRR {fused['mrr']:.3f}")

  [yes] What does MU state that the preparation of financial statements an
  [yes] What termination fee is AMD required to pay ZT Systems under certa
  [yes] What factors does NVIDIA believe are the principal competitive fac
  [yes] What competitive pressures from Intel and Nvidia did AMD face in f
  [yes] What forms the basis for AMD's estimates and judgments used in pre
  [yes] What financial charge did NVIDIA incur in fiscal year 2026 due to 

filter       recall@5 1.00   MRR 0.750
filter+rrf   recall@5 1.00   MRR 0.917


This barely moves, or moves the wrong way. On the full 14 questions fusion
holds recall at 1.00 and takes MRR slightly down, 0.750 to 0.696.

That is a real result. The filter already narrowed the search to one filing.
Inside one filing there is not much left for a second phrasing to find, and
fusion rewards agreement, which can push a single right passage down a place.

Fusion pays off when one phrasing is not enough. Section 7 has those
questions. Five calls per question for nothing is also a result.

## 5. Strategy three: Rerank the shortlist

### What a cross-encoder does differently

Everything so far compares two vectors that were computed separately. The
question was embedded on its own, every chunk was embedded months of
model-time ago, and the comparison is a dot product between them. The
question and the passage never met.

A cross-encoder reads the pair together, question and passage in one forward pass, and scores how well that passage answers that question. Attention runs across both, so "fiscal 2026" in the question can attend directly to the dates in the passage. It is much more accurate, and much too slow to run over two thousand chunks.

So it goes last, over a shortlist. Retrieve a pool of ten cheaply, then spend the cross-encoder on reordering those ten.

This also tells you what reranking cannot do. It only reorders what it is
given. If the right passage is not in the pool, no reranker will find it, and that is why **reranking changes MRR and not recall (remember this)**. Pay attention to exactly that in the numbers.

### TODO 2.3 — rerank (2 pts)

In `rag/retrieval.py`, write `rerank_cohere` and
`filter_rrf_rerank`. 

Two details. Send the **original question** to the reranker, not `f.search`: a cross-encoder reads language, and the question is what the user wrote. And `filter_rrf_rerank` should call `rerank(...)`, the name the next cell gives `rerank_cohere`, so the same code works here and in the module.

**Steps:**

1. `rerank_cohere(question, passages, top_n)`: `cohere.ClientV2(os.environ["COHERE_API_KEY"]).rerank(model="rerank-v3.5", query=question, documents=[text for _, text in passages], top_n=top_n)`. Each result has `.index` into the list you passed; return `passages[r.index]` for each.
2. `filter_rrf_rerank(question, k)`: the same as `filter_rrf` up to the fusion, but keep `config.POOL` candidates instead of `k`. Then `rerank(question, pool, top_n=k)`, with the original question.


In [17]:
import os
def rerank_cohere(question, passages, top_n):
    # TODO 2.3 — cohere.ClientV2(...).rerank(model="rerank-v3.5", ...).
    # It returns results carrying .index into the list you gave it.
    # your code here
    import cohere

    if not passages:
        return []
    client = cohere.ClientV2(os.environ["COHERE_API_KEY"])
    response = client.rerank(
        model="rerank-v3.5",
        query=question,
        documents=[text for _, text in passages],
        top_n=min(top_n, len(passages)),
    )
    return [passages[r.index] for r in response.results]

In [18]:
# See it work on one question before scoring anything.
# This is the clearest single demonstration in the assignment.
f = extract_filter(question)
pool = rrf([retrieval.dense(q, 5, where=where_of(f)) for q in write_queries(f.search)])[:8]

print("pool, in fusion order:")
for i, (source, text) in enumerate(pool):
    print(f"  {i}. {source}  |  {' '.join(text.split())[:70]}")

# the function you just wrote
rerank = rerank_cohere

print()
print("after reranking:")
for i, (source, text) in enumerate(rerank(question, pool, top_n=5)):
    print(f"  {i}. {source}  |  {' '.join(text.split())[:70]}")

pool, in fusion order:
  0. NVDA_2026_risk_factors.txt  |  The increasingly complex export controls impose complex and burdensome
  1. NVDA_2026_risk_factors.txt  |  Given the increasing strategic importance of AI and rising geopolitica
  2. NVDA_2026_risk_factors.txt  |  As a result, export controls have in the past and may in the future ne
  3. NVDA_2026_risk_factors.txt  |  or infer sensitive information about us from public sources, data brok
  4. NVDA_2026_risk_factors.txt  |  could also result in increased taxation, trade sanctions, the impositi
  5. NVDA_2026_risk_factors.txt  |  business and cultural practices, disruptions to capital markets, count
  6. NVDA_2026_risk_factors.txt  |  Government actions, including trade protection and national and econom
  7. NVDA_2026_risk_factors.txt  |  24 Table of Contents

after reranking:
  0. NVDA_2026_risk_factors.txt  |  As a result, export controls have in the past and may in the future ne
  1. NVDA_2026_risk_factors.txt  |  The increa

If your reranker is working, a passage that fusion had at rank 2 or 3 is now
at rank 0, and the ones that were about the right topic in the wrong year have
moved down. The set of passages is nearly the same. The order is better.

In [19]:
# TODO 2.3 — filter_rrf_rerank
def filter_rrf_rerank(question, k=config.K):
    # your code here
    f = extract_filter(question)
    where = where_of(f)
    rankings = [retrieval.dense(q, config.POOL, where=where) for q in write_queries(f.search)]
    pool = rrf(rankings)[:config.POOL]
    return rerank(question, pool, top_n=k)

reranked = evaluate.score_retrieval(filter_rrf_rerank, SAMPLE, k=5, verbose=True)
print()
print(f"filter+rrf          recall@5 {fused['recall@k']:.2f}   MRR {fused['mrr']:.3f}")
print(f"filter+rrf+rerank   recall@5 {reranked['recall@k']:.2f}   MRR {reranked['mrr']:.3f}")

  [yes] What does MU state that the preparation of financial statements an
  [yes] What termination fee is AMD required to pay ZT Systems under certa
  [yes] What factors does NVIDIA believe are the principal competitive fac
  [yes] What competitive pressures from Intel and Nvidia did AMD face in f
  [yes] What forms the basis for AMD's estimates and judgments used in pre
  [yes] What financial charge did NVIDIA incur in fiscal year 2026 due to 

filter+rrf          recall@5 1.00   MRR 0.917
filter+rrf+rerank   recall@5 1.00   MRR 0.889


On the full set this is the best MRR of any strategy, 0.804 against 0.750 for
the filter alone, and recall does not budge from 1.00.

That is the textbook claim about reranking, measured on your own corpus: it
improves the ordering of what you found and it does not find anything new.

### Reranking where there is room to improve

On the six questions above, `filter+rrf` already put the right passage at or
near rank 1 for every question, an MRR close to 1.00. From there a reranker
can only hold the order or disturb it, and a dip of a tenth is one passage
moved one place. That is not the reranker failing. It is the filter having
done the job first.

To see what a reranker does on its own, give it a pool that is not already
sorted. Same six questions, no metadata filter, the top ten by similarity,
reranked down to five.


In [20]:
def baseline_rerank(question, k=config.K):
    pool = retrieval.dense(question, config.POOL)          # no filter, ten candidates
    return rerank(question, pool, top_n=k)

def rrf_rerank(question, k=config.K):                        # fusion without the filter
    pool = rrf([retrieval.dense(q, k) for q in write_queries(question)])[:config.POOL]
    return rerank(question, pool, top_n=k)

plain = evaluate.score_retrieval(retrieval.baseline, SAMPLE, k=5)
plain_r = evaluate.score_retrieval(baseline_rerank, SAMPLE, k=5)
print(f"baseline            recall@5 {plain['recall@k']:.2f}   MRR {plain['mrr']:.3f}")
print(f"baseline+rerank     recall@5 {plain_r['recall@k']:.2f}   MRR {plain_r['mrr']:.3f}")


baseline            recall@5 0.67   MRR 0.325
baseline+rerank     recall@5 0.83   MRR 0.583


Measured with Cohere's `rerank-v3.5`: baseline 0.67 recall and 0.33 MRR, and
with reranking 0.83 and 0.58. MRR nearly doubles. That is the effect the
filtered run hid.

Recall moved too, and it is worth seeing why, because it looks like a
contradiction of the rule above. The pool is ten and `k` is five. A right
passage sitting at rank 7 is inside the pool but outside the top five; the
reranker moves it up and it counts. Reranking still finds nothing new: it
never sees anything beyond the ten. Set `config.POOL` to five and recall
cannot move at all.

Same thing with fusion in place of a single search: `rrf` without the filter
scores 0.50 and 0.33 on these six, and with reranking 0.67 and 0.50.


## 6. One more, given: add a lexical retriever

Embeddings are good at meaning and indifferent to exact strings. Ask for
"HBM3E" and a dense retriever returns passages about high-bandwidth memory in
general. BM25 counts words, so it goes straight to the four places that say
"HBM3E". Run both and fuse them with `rrf`, and you have hybrid retrieval.

`lexical` and `hybrid_rrf` are written for you in `rag/retrieval.py`. Read
them. BM25 needs its own copy of the chunks in
memory, because a vector database will not give you the words back. Two
indexes is the real price of hybrid search.

In [21]:
print("BM25 on an exact term:")
for source, text in retrieval.lexical("HBM3E", 5):
    print(f"  {source}")

print()
print("dense on the same term:")
for source, text in retrieval.dense("HBM3E", 5):
    print(f"  {source}")

BM25 on an exact term:


RuntimeError: Error loading c:\Users\harsh\AgenticAI_Fall26\DADA2\corpus\sections\AMD_2023_business.txt

If the two lists barely overlap, that is the argument for hybrid: two
retrievers that find different things are worth fusing.

In [ ]:
hybrid = evaluate.score_retrieval(retrieval.hybrid_rrf, SAMPLE, k=5, verbose=True)
print()
print(f"filter       recall@5 {filtered['recall@k']:.2f}   MRR {filtered['mrr']:.3f}")
print(f"hybrid+rrf   recall@5 {hybrid['recall@k']:.2f}   MRR {hybrid['mrr']:.3f}")

This one gets worse on these questions, and on the full set it does: 0.93
against 1.00. The reason is in the code you just read. The dense arm is
filtered by metadata and the BM25 arm is not, so half the fusion pulls in
exact-term matches from the wrong filings. Hold that thought for one section.

## 7. The same strategies, harder questions

Everything above was measured on questions that name a company and a year.
Those flatter the filter. Real questions are not all like that. `eval/evalset_hard.json` holds ten that
are not: they compare companies, or span years, or describe a topic without
naming a filing. Each is labeled with several gold files, and we score them
at k=10 because a question needing four files cannot be answered in five
slots.

In [ ]:
comparative = evaluate.load_evalset("comparative")

for item in comparative[:4]:
    print(f"  {item['question']}")
    print(f"    gold: {len(evaluate.gold_of(item))} files\n")

In [ ]:
# Pick one and watch the filter make it worse.
hard = comparative[0]["question"]
print(hard)
print()
f = extract_filter(hard)
print("the extractor decided:", f.ticker, f.fiscal_year)
print("gold files:", evaluate.gold_of(comparative[0]))

The question is about several companies and the extractor picked one. The
filter is about to throw away most of what the question needed. The extractor
did its job. The filter built on it is a confident restriction to the wrong
subset.

Score two strategies here and compare the direction of change with section 3.

In [ ]:
# two strategies, the hard set, k=10. About 30 model calls.
hard_sample = comparative[:5]

hb = evaluate.score_retrieval(retrieval.baseline, hard_sample, k=10)
hf = evaluate.score_retrieval(with_filter, hard_sample, k=10)

print(f"baseline   recall@10 {hb['recall@k']:.2f}   MRR {hb['mrr']:.3f}")
print(f"filter     recall@10 {hf['recall@k']:.2f}   MRR {hf['mrr']:.3f}")

On the full ten questions the filter takes recall **down**, 0.48 to 0.40,
having taken it from 0.71 to 1.00 on the other set. Same code, same index,
opposite conclusion.

And hybrid, which lost in section 6, is the best here by a distance: 0.75
recall where fusion gets 0.58 and the filter gets 0.40. Same reason as before,
other direction. Its BM25 arm is unfiltered, so it reaches across companies,
which is what a question comparing NVIDIA and AMD needs.

There is no best strategy. There is a best strategy per kind of question.
Notebook 03 builds an agent that uses that: it decides which kind a question
is before it searches.

## Before you move on

Move your three answers into `rag/retrieval.py`. Notebook 03 and `app.py`
import `STRATEGIES` from the module.

In [22]:
import importlib
from rag import retrieval as R
importlib.reload(R)

print("strategies:", list(R.STRATEGIES))
print("reranker: cohere")

# every one returns the same shape
for name, search in R.STRATEGIES.items():
    hits = search("What does NVIDIA say about export controls in fiscal 2026?", 3)
    assert isinstance(hits, list) and hits, f"{name} returned nothing"
    assert all(isinstance(h, tuple) and len(h) == 2 for h in hits), \
        f"{name} does not return (source, text) tuples"
    print(f"  {name:22} {len(hits)} hits, top: {hits[0][0]}")

print()
print("all five strategies satisfy the contract")

strategies: ['baseline', 'filter', 'filter+rrf', 'filter+rrf+rerank', 'hybrid+rrf']
reranker: cohere
  baseline               3 hits, top: NVDA_2025_risk_factors.txt
  filter                 3 hits, top: NVDA_2026_risk_factors.txt
  filter+rrf             3 hits, top: NVDA_2026_risk_factors.txt
  filter+rrf+rerank      3 hits, top: NVDA_2026_risk_factors.txt
  hybrid+rrf             3 hits, top: NVDA_2026_risk_factors.txt

all five strategies satisfy the contract
